In [ ]:
from datetime import date

def calculate_use_priority(days_remaining):
    """Rank reminder urgency; this does not determine food safety."""
    if days_remaining is None:
        return "UNKNOWN"
    if days_remaining <= 2:
        return "HIGH"
    if days_remaining <= 5:
        return "MEDIUM"
    return "LOW"


# First check: no storage guidance is being estimated yet.
for days in [None, -1, 0, 2, 3, 5, 6]:
    print(f"Days remaining: {str(days):>4} → {calculate_use_priority(days)}")

In [ ]:
def evaluate_reminder(item_id, reminder_date=None, *, as_of):
    """Use a supplied reminder date without estimating storage duration."""
    days_remaining = (
        (date.fromisoformat(reminder_date) - as_of).days
        if reminder_date is not None else None
    )

    if days_remaining is None:
        status = "MISSING_DATE"
    elif days_remaining < 0:
        status = "PAST_REMINDER"
    elif days_remaining == 0:
        status = "TODAY"
    else:
        status = "UPCOMING"

    return {
        "item_id": item_id,
        "reminder_date": reminder_date,
        "use_priority": calculate_use_priority(days_remaining),
        "days_remaining": days_remaining,
        "reminder_status": status,
    }


# Fixed dates make this example reproducible.
print(evaluate_reminder(
    "demo-001", "2026-10-15", as_of=date(2026, 10, 13)
))
print(evaluate_reminder(
    "demo-002", as_of=date(2026, 10, 13)
))

In [ ]:
expected_priorities = {
    None: "UNKNOWN",
    -1: "HIGH",
    0: "HIGH",
    2: "HIGH",
    3: "MEDIUM",
    5: "MEDIUM",
    6: "LOW",
}

for days, expected in expected_priorities.items():
    assert calculate_use_priority(days) == expected, (
        f"Priority failed for {days} days"
    )

reference_date = date(2026, 10, 13)

date_cases = [
    ("2026-10-12", -1, "PAST_REMINDER"),
    ("2026-10-13", 0, "TODAY"),
    ("2026-10-15", 2, "UPCOMING"),
    (None, None, "MISSING_DATE"),
]

for reminder, expected_days, expected_status in date_cases:
    result = evaluate_reminder(
        "test-item", reminder, as_of=reference_date
    )
    assert result["days_remaining"] == expected_days
    assert result["reminder_status"] == expected_status
    assert result["reminder_date"] == reminder

print("Passed: 7 priority cases and 4 reminder-date cases.")


In [ ]:
from pathlib import Path
import csv
csv_path = Path("data/storage_rules_clean.csv")
if not csv_path.exists(): csv_path = Path("../data/storage_rules_clean.csv")
with csv_path.open(newline="", encoding="utf-8-sig") as f:
    storage_rules = list(csv.DictReader(f))
supported_rule = next(row for row in storage_rules if row["source_record_id"] == "FS-COLD-020")
print("Records loaded:", len(storage_rules))


In [ ]:
supported_rule = next(
    row for row in storage_rules
    if row["source_record_id"] == "FS-COLD-020"
)

for field in [
    "source_record_id",
    "food_group",
    "food_type",
    "storage_location",
    "refrigerator_min_days",
    "refrigerator_max_days",
    "duration_condition",
    "source_url",
]:
    print(f"{field}: {supported_rule[field]}")


## First supported storage rule
- Rule ID: FS-COLD-020
- Food: cooked meat or poultry leftovers
- Storage: refrigerator
- Required date: prepared date
- Dataset guidance: 3–4 days
- Prototype choice: use day 3 as the estimated use-soon reminder.
- Preserve the prepared date and retain the full source range.
- Missing context or unsupported food/storage returns UNKNOWN.

This is a reminder calculation, not a determination of food safety.
The day-3 choice is a prototype policy pending team review.

In [ ]:
supported_rule = next(
    row for row in storage_rules
    if row["source_record_id"] == "FS-COLD-020"
)

print("Rule:", supported_rule["source_record_id"])
print("Minimum days:", supported_rule["refrigerator_min_days"])
print("Maximum days:", supported_rule["refrigerator_max_days"])

In [ ]:
from datetime import timedelta

def evaluate_leftovers(item, *, as_of):
    """Apply FS-COLD-020 only to explicitly confirmed matching items."""
    result = {
        "item_id": item["item_id"],
        "use_priority": "UNKNOWN",
        "days_remaining": None,
        "recommended_use_date": None,
        "prepared_date": item.get("prepared_date"),
        "reminder_status": "UNKNOWN",
        "source": None,
        "reason": "",
    }

    if (
        item.get("food_type") != "Cooked meat or poultry"
        or item.get("storage_location") != "REFRIGERATOR"
    ):
        result["reason"] = "Food or storage is unsupported by this rule."
        return result

    if not item.get("prepared_date"):
        result["reason"] = "Prepared date is required."
        return result

    try:
        prepared = date.fromisoformat(item["prepared_date"])
    except (TypeError, ValueError):
        result["reason"] = "Prepared date must be a valid YYYY-MM-DD date."
        return result

    if prepared > as_of:
        result["reason"] = "Prepared date is in the future; confirm it."
        return result

    reminder = prepared + timedelta(
        days=int(supported_rule["refrigerator_min_days"])
    )
    countdown = evaluate_reminder(
        item["item_id"], reminder.isoformat(), as_of=as_of
    )

    result.update({
        "use_priority": countdown["use_priority"],
        "days_remaining": countdown["days_remaining"],
        "recommended_use_date": reminder.isoformat(),
        "reminder_status": countdown["reminder_status"],
        "source": supported_rule["source_url"],
        "reason": "Estimated day-3 reminder using FS-COLD-020.",
    })
    return result


demo_item = {
    "item_id": "leftovers-001",
    "food_type": "Cooked meat or poultry",
    "storage_location": "REFRIGERATOR",
    "prepared_date": "2026-10-12",
}

print(evaluate_leftovers(demo_item, as_of=date(2026, 10, 13)))

In [ ]:
reference_date = date(2026, 10, 13)

cases = [
    ("Valid leftovers", {}, "HIGH", 2),
    ("Missing prepared date", {"prepared_date": None}, "UNKNOWN", None),
    ("Invalid date", {"prepared_date": "not-a-date"}, "UNKNOWN", None),
    ("Future date", {"prepared_date": "2026-10-14"}, "UNKNOWN", None),
    ("Unsupported storage", {"storage_location": "FREEZER"}, "UNKNOWN", None),
    ("Unsupported food", {"food_type": "Spinach"}, "UNKNOWN", None),
    ("Reminder today", {"prepared_date": "2026-10-10"}, "HIGH", 0),
    ("Past reminder", {"prepared_date": "2026-10-09"}, "HIGH", -1),
]

for label, changes, expected_priority, expected_days in cases:
    item = {**demo_item, **changes}
    original = item.copy()
    result = evaluate_leftovers(item, as_of=reference_date)

    assert result["use_priority"] == expected_priority, label
    assert result["days_remaining"] == expected_days, label
    assert item == original, f"{label}: input was changed"
    assert result["prepared_date"] == item.get("prepared_date"), label

    print(f"PASS: {label}")

print(f"\nPassed: {len(cases)} leftovers cases.")


## Evaluation results — initial prototype
Reference date: 2026-10-13

- Priority cases: 7/7 passed.
- Reminder-date cases: 4/4 passed.
- Leftovers cases: 8/8 passed.
- Total: 19/19 passed.

Checks covered priority boundaries, date countdowns, missing/invalid/future
prepared dates, unsupported food/storage, and preservation of inputs.

Limitations:
- Automatic storage guidance currently supports only refrigerated
  cooked meat or poultry leftovers.
- Non-meat guidance, recall integration, Role 5 ranking integration,
  and user-facing notifications are not yet implemented or tested.
- Passing these tests does not establish food safety.

## Pending team decision: non-meat demo foods
The team has not yet confirmed the demo foods.

Role 3 needs the exact food types, storage locations, and relevant
conditions (such as opened/unopened or whole/cut) before selecting
and verifying additional storage rules.

In [ ]:
import json

sample_items = [
    {**demo_item, "item_id": "sample-001"},
    {
        **demo_item,
        "item_id": "sample-002",
        "prepared_date": None,
    },
    {
        **demo_item,
        "item_id": "sample-003",
        "prepared_date": "2026-10-09",
    },
]

sample_outputs = [
    evaluate_leftovers(item, as_of=date(2026, 10, 13))
    for item in sample_items
]

print(json.dumps(sample_outputs, indent=2))


In [ ]:
print(repr(supported_rule["source_url"]))

In [ ]:
from urllib.parse import urlsplit
def normalize_source_url(value):
    url = value.strip()
    parts = urlsplit(url)
    if parts.scheme != 'https' or not parts.netloc: raise ValueError('Invalid URL')
    return url
clean_url = normalize_source_url(supported_rule["source_url"])
assert clean_url == "https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts"
print("PASS: source is a plain URL.")

## Role 3 Handoff to Samira — Initial Prototype

Owner: Mahdi Mohammadkhani
Status: Working prototype; not yet integration-ready.

### Implemented
- Loads the cleaned 20-record storage dataset.
- Applies rule FS-COLD-020 to refrigerated cooked meat/poultry leftovers.
- Calculates an estimated day-3 reminder from the prepared date.
- Preserves the original prepared date.
- Returns HIGH, MEDIUM, LOW, or UNKNOWN priority.
- Includes 19 passing calculation/test cases and 3 sample outputs.

### Inputs for evaluate_leftovers()
- item_id
- food_type: "Cooked meat or poultry"
- storage_location: "REFRIGERATOR"
- prepared_date: YYYY-MM-DD
- as_of: explicit Python date used for calculation

### Outputs
- item_id
- use_priority
- days_remaining
- recommended_use_date
- prepared_date
- reminder_status
- source
- reason

### Integration boundaries
Role 5 consumes use_priority rather than recreating storage rules.
Recall eligibility is handled by Role 4 and the integration pipeline.
HIGH priority does not establish food safety or recipe eligibility.

### Pending
- Confirm shared field names and demo foods.
- Verify and add selected non-meat guidance.
- Implement user-facing reminder messages.
- Package the function in the repository module.
- Test the handoff with Role 5 ranking.

### Questions for Samira
1. Can your ranking module consume use_priority, or does it currently
   require expiration_priority?
2. Which non-meat foods and storage conditions will the demo use?
3. How should UNKNOWN and PAST_REMINDER outputs be handled?